# Assignment 2: 4th-Degree Polynomial Regression with PyTorch Autograd
**Author:** Kristian Hudák  
**Course:** AI Olympiad (OAI) Preparation Course

This project implements non-linear polynomial regression from scratch using PyTorch tensors and automatic differentiation (`torch.autograd`):
- Custom PyTorch `Dataset` and `DataLoader` for tabular CSV ingestion
- 4th-degree polynomial hypothesis: $y = a + bx + cx^2 + dx^3 + ex^4$
- Manual gradient descent parameter updates via autograd (`requires_grad=True`)
- Training and validation loss tracking across 2,000 epochs
- Visualization of learning curves and polynomial fitted regression curve vs test ground truth


### Assignment Overview

The objective is to train a 4th-degree polynomial regression model ($y = a + bx + cx^2 + dx^3 + ex^4$) to fit non-linear data from `data_train.csv` and evaluate it on `data_test.csv`.

**Key Requirements:**
1. Implement a custom PyTorch `Dataset` subclass (`CustomDataset`) to load CSV data into tensors.
2. Initialize polynomial weights ($a, b, c, d, e$) with random initialization and autograd enabled.
3. Compute forward pass predictions during training and validation.
4. Update parameters using gradient descent with `LEARNING_RATE` and manual gradient zeroing (`.grad = None`).
5. Log training and validation loss per epoch in `train_losses` and `val_losses`.
6. Plot loss curves and compare model predictions against ground truth test samples.


> **Note:** In Google Colab, upload `data_train.csv` and `data_test.csv` to session storage before execution.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd


class CustomDataset(Dataset):
    def __init__(self, path):
        # Load data from the CSV file pointed to by path
        data = pd.read_csv(path)
        self.x =  torch.tensor(data.iloc[:,:-1].values, dtype = torch.float)
        self.y =  torch.tensor(data.iloc[:,-1].values, dtype = torch.float)
        return

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


In [ ]:
BATCH_SIZE = 42
N_EPOCHS = 2000
LEARNING_RATE = 1e-5

In [ ]:
train_dataset = CustomDataset("/content/data_train.csv")
test_dataset = CustomDataset("/content/data_test.csv")

train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset))

In [ ]:
dtype = torch.float
# Initialize parameters for 4th-degree polynomial: a + b*x + c*x^2 + d*x^3 + e*x^4
a = torch.randn((), dtype=dtype, requires_grad=True)
b = torch.randn((), dtype=dtype, requires_grad=True)
c = torch.randn((), dtype=dtype, requires_grad=True)
d = torch.randn((), dtype=dtype, requires_grad=True)
e = torch.randn((), dtype=dtype, requires_grad=True)

train_losses = []
val_losses = []

for epoch in range(N_EPOCHS):
    epoch_loss = 0.0
    num_batches = 0
    val_loss = 0.0

    # TRAINING LOOP
    for batch in train_loader:
        x, y =  batch

        # Forward pass: compute polynomial y_pred
        y_pred = a + b*x + c*x**2 + d*x**3 + e*x**4



        # Mean Squared Error loss
        loss = (y_pred - y).pow(2).mean()  # batch MSE
        epoch_loss += loss.item()
        num_batches += 1

        # Backward pass (compute gradients)
        loss.backward()

        # Parameter update via gradient descent
        with torch.no_grad():   # Update parameters using LEARNING_RATE and autograd gradients
            a -=  LEARNING_RATE * a.grad
            b -=  LEARNING_RATE * b.grad
            c -=  LEARNING_RATE * c.grad
            d -=  LEARNING_RATE * d.grad
            e -=  LEARNING_RATE * e.grad

            # Zero gradients before next iteration
            a.grad = b.grad = c.grad = d.grad = e.grad = None

    avg_epoch_loss = epoch_loss / num_batches # Epoch average loss
    train_losses.append(avg_epoch_loss)


    # VALIDATION ON TEST DATA
    with torch.no_grad():  # No gradients required for evaluation
      for xb, yb in test_loader:
        # Prediction
        # Forward pass: compute polynomial y_pred
        y_pred = a + b*xb + c*xb**2 + d*xb**3 + e*xb**4

        # Compute MSE loss
        val_loss = torch.mean((y_pred - yb)**2)

        # Record validation loss for this epoch
        val_losses.append(val_loss.item())


    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | loss = {avg_epoch_loss:.6f} | vlidation loss = {val_loss:.6f}")


In [ ]:
# Plot training and validation loss curves

import matplotlib.pyplot as plt

epochs = range(1, N_EPOCHS + 1)

plt.figure(figsize=(8,5))
plt.scatter(train_losses, epochs, color='blue', label='Training Loss', alpha=0.6)
plt.scatter(val_losses, epochs, color='red', label='Validation Loss', alpha=0.6)
plt.title("Training vs Validation Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Plot model predictions vs ground truth test data
# Compares actual test points (x_test, y_test) with polynomial predictions


x_test, y_test = next(iter(test_loader))

with torch.no_grad():
    y_pred = a + b * x_test + c * x_test**2 + d * x_test**3 + e * x_test**4

x_test_np = x_test.numpy()
y_test_np = y_test.numpy()
y_pred_np = y_pred.numpy()

plt.figure(figsize=(8,5))
plt.scatter(x_test_np, y_test_np, color='green', label='Ground Truth (y_test)')
plt.plot(x_test_np, y_pred_np, color='purple', label='Model Prediction Curve', linewidth=2)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Fitted 4th-Degree Polynomial vs Test Ground Truth')
plt.legend()
plt.grid(True)
plt.show()
